In [5]:
import pandas as pd
combined_df = pd.read_pickle('combined_df_all_databases_after_duplicate_2.pkl')
combined_df

,PubMed ID,DOI,Title,Year,Abstract,Author Keywords
0,16985399.0,10.1097/00001786-200610000-00006,Missed nursing care: a qualitative study,2006,The purpose of this study was to determine nur...,[]
1,15946170.0,10.1111/j.1365-2934.2005.00541.x,Confirming mentorship,2005,Mentorship is related to nurses' success in nu...,[]
2,17147757.0,10.1111/j.1741-6787.2004.04008.x,Methods in translation science,2004,The gap between research findings and practice...,[]
3,19076465.0,10.1111/j.1744-6198.2008.00115.x,Competency: a concept analysis,2008,Concept analysis of competency.\nThe purpose o...,[]
4,15946274.0,10.1111/j.1365-2702.2005.01136.x,Clinical education facilitators: a literature ...,2005,"The aim of this literature review, set within ...",[]
...,...,...,...,...,...,...
16926,NaN,NaN,Enhancing Nursing Practice in Developmental Ca...,2008,This study aimed to enhance nursing practice i...,developmental care; participatory action resea...
16927,NaN,NaN,Project Evidencia [evidence] - research and ed...,2013,Project Evidencia [Evidence] intends to promot...,"Evidence-based nursing; Databases, bibliograph..."
16928,NaN,NaN,Use of Research by Undergraduate Nursing Stude...,2017,Research utilization (RU) is crucial to prepar...,Nursing; Students; Nursing Research; Research ...
16929,NaN,NaN,Effects of an Eating Behaviour Modification Pr...,2013,The purpose of this study was to evaluate the ...,Eating behaviour; diabetes; obesity; modificat...


In [6]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16931 entries, 0 to 16930
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   PubMed ID        16805 non-null  float64
 1   DOI              15258 non-null  object 
 2   Title            16931 non-null  object 
 3   Year             16931 non-null  int64  
 4   Abstract         16028 non-null  object 
 5   Author Keywords  16634 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 793.8+ KB


In [7]:
!pip install spacy-langdetect
!pip install spacy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.1/998.1 kB 40.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.7-py3-none-any.whl size=993412 sha256=ac39f52cadcf81def13dd9b23c10734ab3ffd23cdcd3ff869de1d1d31e360e23
  Stored in directory: /root/.cache/pip/wheels/c7/22/8b/c053f2c06a804491c614f194322b8fa897fd092e47e75c51de
Successfully built langdetect


In [8]:
import spacy
from spacy.language import Language
from spacy_langdetect import LanguageDetector

# Define a function to create the LanguageDetector component
@Language.factory("language_detector")
def create_language_detector(nlp, name):
    return LanguageDetector()

# Load a SpaCy model
nlp = spacy.load("en_core_web_sm")  # Ensure you've downloaded this model

# Add the language detector to the pipeline
nlp.add_pipe("language_detector", last=True)



In [9]:
def detect_language(text):
    """
    Detects the language of the given text.

    Args:
        text (str): The input text to analyze.

    Returns:
        dict: A dictionary with the detected language and confidence score.
    """
    doc = nlp(text)
    return doc._.language['language']

'he'

#check English

In [10]:
print(combined_df.shape)
filtered_df = combined_df[combined_df['Abstract'].notnull()]
print(filtered_df.shape)
filtered_df = filtered_df[filtered_df['Abstract'] != '[No abstract available]']
print(filtered_df.shape)

(16931, 6)
(16028, 6)
(15999, 6)


In [11]:
from tqdm import tqdm

tqdm.pandas()

filtered_df['Abstract']  = filtered_df['Abstract'].apply(lambda x: str(x))
# filtered_df['english']  = filtered_df['Abstract'].apply(detect_language)

In [12]:
from tqdm import tqdm

# Enable progress tracking for pandas
tqdm.pandas()

# Apply the detect_language function with a progress bar
filtered_df['english'] = filtered_df['Abstract'].progress_apply(lambda x: detect_language(x))


100%|██████████| 15999/15999 [11:54<00:00, 22.40it/s]


In [13]:
filtered_df['english'].value_counts()

,count
english,
en,15935
es,27
fr,20
de,5
it,4
ca,3
sl,2
pt,2
da,1


In [14]:
tmp = filtered_df[filtered_df['english'] =='en']
tmp.shape

(15935, 7)

In [33]:
filtered_df = tmp[
    ~(
        (tmp['DOI'] == '10.4040/jkan.2016.46.4.501') |
        (tmp['DOI'] == '10.4040/jkan.2004.34.5.838') |
        (tmp['DOI'] == '10.1016/j.enfcli.2017.03.015') |
        (tmp['Title'] == "Nurses' perceptions of barriers to nursing research utilization in clinical environment in Zanjan hospitals, 2010")
    )
]


In [35]:
filtered_df.reset_index(drop=True, inplace=True)
filtered_df

,PubMed ID,DOI,Title,Year,Abstract,Author Keywords,english
0,16985399.0,10.1097/00001786-200610000-00006,Missed nursing care: a qualitative study,2006,The purpose of this study was to determine nur...,[],en
1,15946170.0,10.1111/j.1365-2934.2005.00541.x,Confirming mentorship,2005,Mentorship is related to nurses' success in nu...,[],en
2,17147757.0,10.1111/j.1741-6787.2004.04008.x,Methods in translation science,2004,The gap between research findings and practice...,[],en
3,19076465.0,10.1111/j.1744-6198.2008.00115.x,Competency: a concept analysis,2008,Concept analysis of competency.\nThe purpose o...,[],en
4,15946274.0,10.1111/j.1365-2702.2005.01136.x,Clinical education facilitators: a literature ...,2005,"The aim of this literature review, set within ...",[],en
...,...,...,...,...,...,...,...
15926,NaN,NaN,Enhancing Nursing Practice in Developmental Ca...,2008,This study aimed to enhance nursing practice i...,developmental care; participatory action resea...,en
15927,NaN,NaN,Project Evidencia [evidence] - research and ed...,2013,Project Evidencia [Evidence] intends to promot...,"Evidence-based nursing; Databases, bibliograph...",en
15928,NaN,NaN,Use of Research by Undergraduate Nursing Stude...,2017,Research utilization (RU) is crucial to prepar...,Nursing; Students; Nursing Research; Research ...,en
15929,NaN,NaN,Effects of an Eating Behaviour Modification Pr...,2013,The purpose of this study was to evaluate the ...,Eating behaviour; diabetes; obesity; modificat...,en


In [30]:
# tmp[tmp['DOI']=='10.4040/jkan.2016.46.4.501']
# tmp[tmp['DOI']=='10.4040/jkan.2004.34.5.838']
# tmp[tmp['DOI']=='10.1016/j.enfcli.2017.03.015']
# tmp[tmp['Title']=="Nurses' perceptions of barriers to nursing research utilization in clinical environment in Zanjan hospitals, 2010"]

In [36]:
filtered_df.to_pickle('combined_df_all_databases_only_english_3.pkl')